# Método dos elementos finitos na região Mainback

Este notebook aplica um resolvedor de elementos finitos ao painel traseiro do produto, a região Mainback, e mede se a geometria suposta descreve o que os termopares registraram. A região foi escolhida pela investigação de melhoria do conjunto de dados: com o piso de ruído revisado para 0,57 grau contra erro de 5,68 graus, ela tem a maior folga relativa do projeto, e cinco dos seus dez pontos são sólidos passivos aquecidos por condução a partir da cavidade do forno, que é o caso em que o método se aplica.

O painel traseiro é, segundo o material de kickoff da parceira, o painel estrutural que abriga a exaustão, o cabeamento de potência e os motores dos ventiladores. Fica atrás da cavidade do forno, separado dela pelo isolamento em lã de vidro.

## O que este notebook testa, e o critério declarado antes de rodar

A pergunta não é se o resolvedor funciona, e sim se a geometria suposta descreve a peça. O teste decisivo é de representabilidade: concedidos ao modelo físico os escalares verdadeiros de cada ensaio, quanto ele reproduz das leituras observadas, comparado à base estimada dos próprios dados com os mesmos graus de liberdade.

O critério para prosseguir à etapa preditiva, declarado aqui antes da execução, é que a base do FEM explique, nos cinco pontos de sólido passivo, ao menos tanta variância entre componentes quanto a base dos dados com o mesmo número de modos. Se o critério não for atingido, o notebook encerra com o veredito e com a informação que o destravaria.

Nenhuma linha de dado é criada. O conjunto de teste reservado não é consultado.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/colabs_FEM"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import json
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve

BASE = "../../"
PREFIXOS = ("blower_direction_", "housing_", "board_position_", "run_case_", "losses_",
            "perfil_perdas_", "deflector_hole_", "material_coil_support_", "cfg_",
          "reason_", "loss_profile_order_")

dados = pd.read_csv(BASE + "dataset_mainback_13cat.csv")
alvos = [c for c in dados.columns if not c.startswith(PREFIXOS)]
print(f"ensaios: {len(dados)} | componentes: {len(alvos)}")
for c in alvos:
    print(f"   {c:<36} media {dados[c].mean():6.1f} C  desvio {dados[c].std():5.1f} C  ausentes {int(dados[c].isna().sum())}")

## 1. Tabela de suposições declaradas

A parceira ainda não forneceu geometria, materiais nem posições dos termopares. As suposições abaixo sustentam o modelo e estão registradas com origem e faixa, de modo que nenhum valor suposto seja apresentado como medido.

A classificação de cada ponto segue o documento de classificação dos componentes. Os pontos que medem ar não são alvo do método, e os que geram calor próprio, o enrolamento do motor de convecção e o cabo de potência, tampouco. Os cinco de sólido passivo são o alvo legítimo.

In [ ]:
suposicoes = pd.DataFrame([
    {"grandeza": "material do painel", "adotado": "aco carbono, k = 45 W/(m.K)",
     "origem": "painel estrutural estampado, kickoff p. 8", "faixa": "aco inox k = 16 a aco carbono k = 55"},
    {"grandeza": "espessura", "adotado": "1,0 mm", "origem": "chapa estrutural de eletrodomestico", "faixa": "0,7 a 1,5 mm"},
    {"grandeza": "dimensoes do painel", "adotado": "0,76 m por 0,60 m", "origem": "fogao de 30 polegadas, altura da secao do forno", "faixa": "nao aplicavel"},
    {"grandeza": "fonte: pegada da cavidade", "adotado": "retangulo central, 0,50 por 0,40 m, a partir de 0,08 m da base",
     "origem": "cavidade do forno atras do painel", "faixa": "posicao vertical variada na Secao 5"},
    {"grandeza": "acoplamento com a cavidade", "adotado": "Robin com h_iso = 3 W/(m2.K) para T_cav",
     "origem": "conducao atraves de la de vidro de 50 mm", "faixa": "2 a 6"},
    {"grandeza": "conveccao na face externa", "adotado": "h = 8 W/(m2.K)", "origem": "conveccao natural em painel vertical", "faixa": "5 a 12; 30 junto ao soprador"},
    {"grandeza": "posicoes dos termopares", "adotado": "declaradas na celula seguinte", "origem": "pendente junto a parceira", "faixa": "nao aplicavel"},
])
print(suposicoes.to_string(index=False))

# classe de cada ponto, conforme documents/extras/relatorios_sprint_4/classificacaoComponentesFEM.md
CLASSE = {
    "Door Lock Motor Ambient 1/2\" Below": "ar",
    "Door Lock Microswitch Body": "passivo",
    "Convect Motor Winding": "enrolamento",
    "Hot Surface Indicator Body": "passivo",
    "Power On Indicator Body": "passivo",
    "Storage Drawer (Metal Above)": "passivo",
    "L2, Wire, Power Cord": "cabo",
    "Ambient Below Terminal Block": "ar",
    "ACU Ambient (Hill Only) Top": "ar",
    "ACU K8 Housing / Double Line Break": "passivo",
}
# posicoes supostas no painel, em metros: x da esquerda, y da base
SONDAS = {
    "Door Lock Microswitch Body": (0.38, 0.55),
    "Door Lock Motor Ambient 1/2\" Below": (0.38, 0.52),
    "Hot Surface Indicator Body": (0.12, 0.56),
    "Power On Indicator Body": (0.16, 0.56),
    "Convect Motor Winding": (0.38, 0.30),
    "Storage Drawer (Metal Above)": (0.38, 0.04),
    "L2, Wire, Power Cord": (0.66, 0.08),
    "Ambient Below Terminal Block": (0.66, 0.05),
    "ACU K8 Housing / Double Line Break": (0.66, 0.54),
    "ACU Ambient (Hill Only) Top": (0.66, 0.58),
}
PASSIVOS = [c for c in alvos if CLASSE.get(c) == "passivo"]
print("\npontos de solido passivo, alvo legitimo do metodo:", PASSIVOS)

## 2. O resolvedor

Chapa plana em regime permanente, com condução no plano, perda convectiva pela face externa e, na pegada da cavidade, acoplamento de Robin à temperatura da cavidade através do isolamento. Fora da pegada, a face interna é tratada como adiabática. As arestas trocam por convecção.

Para coeficientes fixos, a solução é afim na temperatura da cavidade e na do ar, de modo que um único ajuste fornece o perfil espacial de cada fonte, e qualquer par de temperaturas é obtido sem resolver de novo.

In [ ]:
LARGURA, ALTURA = 0.76, 0.60
ESPESSURA = 0.001
K_ACO = 45.0
H_EXT = 8.0
H_ISO = 3.0
PEGADA = (0.13, 0.63, 0.08, 0.48)      # x0, x1, y0, y1 da pegada da cavidade


def malha(nx, ny, largura=LARGURA, altura=ALTURA):
    xs, ys = np.linspace(0, largura, nx + 1), np.linspace(0, altura, ny + 1)
    X, Y_ = np.meshgrid(xs, ys, indexing="ij")
    p = np.column_stack([X.ravel(), Y_.ravel()])
    pos = lambda i, j: i * (ny + 1) + j
    t = []
    for i in range(nx):
        for j in range(ny):
            a, b, c, d = pos(i, j), pos(i + 1, j), pos(i + 1, j + 1), pos(i, j + 1)
            t.append([a, b, c]); t.append([a, c, d])
    return p, np.array(t)


def na_pegada(cx, cy, pegada=PEGADA):
    x0, x1, y0, y1 = pegada
    return (x0 <= cx <= x1) and (y0 <= cy <= y1)


def monta(p, t, h_ext=H_EXT, h_iso=H_ISO, pegada=PEGADA, k=K_ACO, espessura=ESPESSURA):
    """Devolve A, e os fatores de carga separados: do ar externo e da cavidade."""
    n = len(p)
    A = sp.lil_matrix((n, n))
    f_ar, f_cav = np.zeros(n), np.zeros(n)
    for tri in t:
        x, y_ = p[tri, 0], p[tri, 1]
        area = 0.5 * ((x[1] - x[0]) * (y_[2] - y_[0]) - (x[2] - x[0]) * (y_[1] - y_[0]))
        gx = np.array([y_[1] - y_[2], y_[2] - y_[0], y_[0] - y_[1]]) / (2 * area)
        gy = np.array([x[2] - x[1], x[0] - x[2], x[1] - x[0]]) / (2 * area)
        A[np.ix_(tri, tri)] += k * espessura * area * (np.outer(gx, gx) + np.outer(gy, gy))
        massa = area / 12.0 * (np.ones((3, 3)) + np.eye(3))
        A[np.ix_(tri, tri)] += h_ext * massa
        f_ar[tri] += h_ext * area / 3.0
        if na_pegada(x.mean(), y_.mean(), pegada):
            A[np.ix_(tri, tri)] += h_iso * massa
            f_cav[tri] += h_iso * area / 3.0
    arestas = []
    for tri in t:
        for a, c in ((tri[0], tri[1]), (tri[1], tri[2]), (tri[2], tri[0])):
            arestas.append((min(a, c), max(a, c)))
    for (a, c), q in Counter(arestas).items():
        if q == 1:
            L = np.hypot(p[c, 0] - p[a, 0], p[c, 1] - p[a, 1])
            A[np.ix_([a, c], [a, c])] += h_ext * espessura * L * np.array([[1/3, 1/6], [1/6, 1/3]])
            f_ar[[a, c]] += h_ext * espessura * L / 2
    return A.tocsr(), f_ar, f_cav


def resolve(nx, ny, T_cav, T_ar, **kw):
    p, t = malha(nx, ny)
    A, f_ar, f_cav = monta(p, t, **kw)
    return p, t, spsolve(A, f_ar * T_ar + f_cav * T_cav)


def interpola(p, t, u, ponto):
    px, py = ponto
    for tri in t:
        x, y_ = p[tri, 0], p[tri, 1]
        det = (y_[1] - y_[2]) * (x[0] - x[2]) + (x[2] - x[1]) * (y_[0] - y_[2])
        l0 = ((y_[1] - y_[2]) * (px - x[2]) + (x[2] - x[1]) * (py - y_[2])) / det
        l1 = ((y_[2] - y_[0]) * (px - x[2]) + (x[0] - x[2]) * (py - y_[2])) / det
        if min(l0, l1, 1 - l0 - l1) >= -1e-9:
            return l0 * u[tri[0]] + l1 * u[tri[1]] + (1 - l0 - l1) * u[tri[2]]
    return float("nan")


print("resolvedor definido")

## 3. Verificação

Convergência com o refino da malha, e controle contra a solução analítica da aleta numa faixa estreita aquecida pela base. Sem as duas, um resolvedor que converge para a resposta errada passaria despercebido.

In [ ]:
print("VERIFICACAO 1: convergencia de malha (cavidade a 180 C, ar a 25 C)")
print(f"{'nx x ny':>10}{'nos':>7}{'elementos':>11}{'T na gaveta':>13}{'T no ACU':>10}{'variacao':>10}")
ant = None
for nx, ny in [(19, 15), (38, 30), (76, 60), (114, 90)]:
    p, t, u = resolve(nx, ny, 180.0, 25.0)
    tg = interpola(p, t, u, SONDAS["Storage Drawer (Metal Above)"])
    ta = interpola(p, t, u, SONDAS["ACU K8 Housing / Double Line Break"])
    print(f"{nx:>6} x {ny:<3}{len(p):>7}{len(t):>11}{tg:>13.3f}{ta:>10.3f}{'' if ant is None else f'{abs(tg-ant):>10.4f}'}")
    ant = tg
NX, NY = 76, 60

print("\nVERIFICACAO 2: controle analitico, faixa estreita com a base a T_base e sem cavidade")
h = H_EXT
# Este resolvedor perde calor por UMA face (a interna e adiabatica fora da pegada),
# de modo que a aleta de controle usa m = raiz(h / (k e)), e nao 2h como no console.
m = np.sqrt(h / (K_ACO * ESPESSURA))
print(f"  comprimento caracteristico 1/m = {1000 / m:.1f} mm")
# faixa de 0,02 x 0,30 m, base a 100 C: impoe Dirichlet na base
p, t = malha(4, 60, 0.02, 0.30)
A, f_ar, _ = monta(p, t, pegada=(9, 9, 9, 9))          # pegada fora do dominio: sem cavidade
base = np.where(np.abs(p[:, 1]) < 1e-12)[0]
livres = np.setdiff1d(np.arange(len(p)), base)
u = np.zeros(len(p)); u[base] = 100.0
u[livres] = spsolve(A[livres][:, livres].tocsc(), (f_ar * 25.0)[livres] - A[livres][:, base] @ u[base])
for y_ in (0.05, 0.10, 0.20):
    fem = interpola(p, t, u, (0.01, y_))
    ana = 25.0 + 75.0 * np.cosh(m * (0.30 - y_)) / np.cosh(m * 0.30)
    print(f"  y={y_*1000:>5.0f} mm   FEM {fem:>7.3f}   analitico {ana:>7.3f}   diferenca {abs(fem-ana):.4f} C")

## 4. Perfis espaciais das duas fontes, e o campo típico

Com a cavidade a 1 e o ar a 0, o campo é o perfil da cavidade; com o ar a 1 e a cavidade a 0, o perfil do ar. Qualquer ensaio é combinação dos dois.

In [ ]:
p, t, u_cav = resolve(NX, NY, 1.0, 0.0)
_, _, u_ar = resolve(NX, NY, 0.0, 1.0)
phi_cav = np.array([interpola(p, t, u_cav, SONDAS[c]) for c in alvos])
phi_ar = np.array([interpola(p, t, u_ar, SONDAS[c]) for c in alvos])
print(f"{'componente':<36}{'classe':<12}{'phi cavidade':>13}{'phi ar':>9}")
for c, a, b in zip(alvos, phi_cav, phi_ar):
    print(f"{c:<36}{CLASSE.get(c, '?'):<12}{a:>13.4f}{b:>9.4f}")
print(f"\namplitude do perfil da cavidade entre os pontos passivos: "
      f"{max(phi_cav[alvos.index(c)] for c in PASSIVOS) - min(phi_cav[alvos.index(c)] for c in PASSIVOS):.4f}")

## 5. Representabilidade, com graus de liberdade iguais

As leituras observadas, centradas por componente, são projetadas na base do FEM e na base estimada dos dados, cada uma com dois escalares por ensaio. A base do FEM recebe também uma segunda parametrização, com a pegada da cavidade deslocada verticalmente, para que a fonte não uniforme seja representável. O resíduo e a variância explicada de cada base são comparados nos dez pontos e nos cinco passivos.

In [ ]:
def base_fem(pegadas, h_ext=H_EXT):
    """Perfis nas sondas para cada pegada de cavidade fornecida."""
    formas = []
    for peg in pegadas:
        p, t = malha(NX, NY)
        A, f_ar, f_cav = monta(p, t, h_ext=h_ext, pegada=peg)
        u = spsolve(A, f_cav)
        formas.append(np.array([interpola(p, t, u, SONDAS[c]) for c in alvos]))
    return np.vstack(formas)


def representabilidade(Y, G_fem, rotulo):
    mu = Y.mean(0)
    C = Y - mu
    Gc = G_fem - G_fem.mean(1, keepdims=True)
    proj_fem = C @ np.linalg.pinv(Gc) @ Gc
    U, S, Vt = np.linalg.svd(C, full_matrices=False)
    k = G_fem.shape[0]
    proj_dados = U[:, :k] @ np.diag(S[:k]) @ Vt[:k]
    var = lambda pr: 100 * (np.linalg.norm(pr) ** 2) / (np.linalg.norm(C) ** 2)
    print(f"{rotulo}  (n={len(Y)} ensaios completos, {Y.shape[1]} pontos, {k} escalares por ensaio)")
    print(f"   base do FEM      MAE residual {np.abs(C - proj_fem).mean():6.3f} C   variancia explicada {var(proj_fem):5.1f}%")
    print(f"   base dos dados   MAE residual {np.abs(C - proj_dados).mean():6.3f} C   variancia explicada {var(proj_dados):5.1f}%")
    print(f"   so a media       MAE residual {np.abs(C).mean():6.3f} C")
    return var(proj_fem), var(proj_dados)


PEGADAS = [PEGADA, (0.13, 0.63, 0.20, 0.60)]           # pegada adotada, e deslocada para o alto
G = base_fem(PEGADAS)
print("perfis do FEM nas sondas (pegada adotada e pegada alta):")
for c, a, b in zip(alvos, G[0], G[1]):
    print(f"   {c:<36}{a:>8.4f}{b:>8.4f}")
print()
Y_tudo = dados[alvos].dropna().to_numpy(float)
idx_p = [alvos.index(c) for c in PASSIVOS]
Y_pas = dados[PASSIVOS].dropna().to_numpy(float)
vf10, vd10 = representabilidade(Y_tudo, G, "DEZ PONTOS")
print()
vf5, vd5 = representabilidade(Y_pas, G[:, idx_p], "CINCO PONTOS PASSIVOS")
PISO_REVISADO = 0.57
print(f"\npiso de ruido revisado da regiao: {PISO_REVISADO:.2f} C")
CRITERIO = vf5 >= vd5
print(f"\nCRITERIO DECLARADO (FEM explica ao menos tanto quanto os dados, nos passivos): {'ATINGIDO' if CRITERIO else 'NAO ATINGIDO'}")

## 6. Sensibilidade das suposições

Antes de concluir, verifica-se se alguma das incertezas declaradas moveria o resultado. Cada grandeza é variada dentro da sua faixa, e o efeito sobre a representabilidade nos pontos passivos é registrado.

In [ ]:
print(f"{'variacao':<44}{'var. FEM passivos':>18}")
for rot, kw, pegs in [
    ("referencia", {}, PEGADAS),
    ("aco inox, k = 16", {"k": 16.0}, PEGADAS),
    ("espessura 1,5 mm", {"espessura": 0.0015}, PEGADAS),
    ("h externo 12", {"h_ext": 12.0}, PEGADAS),
    ("h isolamento 6", {"h_iso": 6.0}, PEGADAS),
    ("pegada estreita (0,25 a 0,51 m)", {}, [(0.25, 0.51, 0.08, 0.48), (0.25, 0.51, 0.20, 0.60)]),
    ("pegada ate o topo", {}, [(0.13, 0.63, 0.08, 0.60), (0.13, 0.63, 0.30, 0.60)]),
]:
    formas = []
    for peg in pegs:
        p, t = malha(NX, NY)
        A, f_ar, f_cav = monta(p, t, pegada=peg, **kw)
        u = spsolve(A, f_cav)
        formas.append(np.array([interpola(p, t, u, SONDAS[c]) for c in PASSIVOS]))
    Gp = np.vstack(formas)
    C = Y_pas - Y_pas.mean(0)
    Gc = Gp - Gp.mean(1, keepdims=True)
    pr = C @ np.linalg.pinv(Gc) @ Gc
    print(f"{rot:<44}{100 * np.linalg.norm(pr) ** 2 / np.linalg.norm(C) ** 2:>17.1f}%")

## 6.1. Leitura física dos perfis

Os perfis da Seção 4 mostram que, na geometria suposta, a cavidade responde por apenas três a onze por cento do excesso de temperatura nos pontos passivos, e o ar pelo restante. As leituras, porém, situam esses pontos dezenas de graus acima do ambiente. O acoplamento com a cavidade através do isolamento é, portanto, fraco demais para explicar o observado, e a variação do coeficiente de isolamento na Seção 6 não altera a representabilidade.

A leitura consistente com a classificação dos componentes é que o calor alcança os pontos do painel traseiro por outro caminho, não representado nesta geometria: o ar de exaustão, que sai da cavidade e percorre o painel, e os motores dos ventiladores, que geram calor próprio. Três dos dez pontos medem exatamente esse ar, e um mede o enrolamento do motor. Um modelo físico útil para esta região precisaria acoplar o escoamento de exaustão ao painel, o que é problema de dinâmica de fluidos, e não de condução em sólido.

## 7. Veredito

O veredito é calculado na célula seguinte a partir do critério declarado na abertura, e não escrito a mão.

In [ ]:
print("VEREDITO")
if CRITERIO:
    print("  A base derivada do FEM representa os pontos passivos ao menos tao bem quanto a base dos dados.")
    print("  A etapa preditiva, prever os escalares da cavidade e do ar a partir da configuracao, esta autorizada.")
else:
    print(f"  Nos cinco pontos passivos, a base do FEM explica {vf5:.1f}% da variancia entre componentes, contra")
    print(f"  {vd5:.1f}% da base estimada dos proprios dados com o mesmo numero de escalares por ensaio.")
    print("  A geometria suposta nao descreve a peca: a substituicao da formulacao entregue pelo FEM nao e")
    print("  recomendada enquanto a parceira nao fornecer as posicoes dos termopares e a geometria do painel.")
    print("  O resolvedor permanece verificado e registrado para ser reaproveitado quando a geometria chegar.")